# 03 dbutils

## Goal

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 Goal</b><br>
Use the Databricks utilities (<code>dbutils</code>) from notebooks: <code>fs</code> for files in volumes, <code>widgets</code> for parameters, <code>secrets</code> for credentials, <code>notebook</code> for running and exiting notebooks, and <code>jobs.taskValues</code> for passing values between job tasks.
</div>

## Why it matters

<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Why it matters</b><br>
<code>dbutils</code> is the glue between notebooks and the platform: listing and moving files, reading job parameters, fetching secrets, and returning results to orchestrators. Every Databricks data engineer uses it daily, and the exam expects you to recognize its main modules.
</div>

## Concept

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 The dbutils modules</b><br>
| Module | Purpose | Example |<br>|---|---|---|<br>| <code>dbutils.fs</code> | File operations on volumes and cloud paths | <code>dbutils.fs.ls("/Volumes/...")</code> |<br>| <code>dbutils.widgets</code> | Notebook parameters (input boxes, dropdowns) | <code>dbutils.widgets.get("run_date")</code> |<br>| <code>dbutils.secrets</code> | Read secrets from secret scopes (values are redacted in output) | <code>dbutils.secrets.get("scope", "key")</code> |<br>| <code>dbutils.notebook</code> | Run another notebook, or exit with a value | <code>dbutils.notebook.run("./child", 600, {...})</code> |<br>| <code>dbutils.jobs.taskValues</code> | Pass small values between tasks of a job | <code>dbutils.jobs.taskValues.set("rows", 42)</code> |<br>| <code>dbutils.library</code> | (Legacy) restart Python | <code>dbutils.library.restartPython()</code> |<br><br>
<code>dbutils.help()</code> and <code>dbutils.fs.help()</code> print the full reference.
</div>

## Real-world example

<div style="background:#FFF4DB;border-left:5px solid #E5A100;padding:10px 14px;border-radius:6px;color:#4a3500;">
<b>🏦 Real-world example</b><br>
A daily job's first task lists new files in a landing volume with <code>dbutils.fs.ls</code>, reads the processing date from a widget set by the job, loads the files, and publishes the number of rows loaded with <code>dbutils.jobs.taskValues.set</code>. The next task reads that value and skips the expensive Gold rebuild when nothing new arrived. Credentials for the source API come from <code>dbutils.secrets.get</code>, so they never appear in code or logs.
</div>

## Setup

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Creates a volume for this lesson's files.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
The base path of the lesson's folder in the volume.
</div>

In [0]:
spark.sql("CREATE SCHEMA IF NOT EXISTS workspace.platform")
spark.sql("CREATE VOLUME IF NOT EXISTS workspace.platform.files")
BASE = "/Volumes/workspace/platform/files/lesson03"
dbutils.fs.mkdirs(BASE)
print("BASE =", BASE)

## 1. dbutils.fs: work with files

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Writes small files, lists them, reads one back, copies, moves and deletes files, all in a Unity Catalog volume.<br><br>
<b>Why it matters</b><br>Ingestion code constantly manages files: checking what arrived, archiving processed files, cleaning up temporary folders. <code>dbutils.fs</code> works on volumes (<code>/Volumes/...</code>) and, with permissions, on cloud storage paths.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
A listing with 3 files and their sizes, the first lines of one file, then the archive folder containing the moved file.
</div>

In [0]:
for i in range(3):
    dbutils.fs.put(f"{BASE}/landing/orders_{i}.csv", f"order_id,amount\n{i},{i * 10}\n", overwrite=True)

for f in dbutils.fs.ls(f"{BASE}/landing"):
    print(f"{f.name:<16} {f.size:>4} bytes  modified={f.modificationTime}")

print(dbutils.fs.head(f"{BASE}/landing/orders_1.csv"))

dbutils.fs.cp(f"{BASE}/landing/orders_0.csv", f"{BASE}/backup/orders_0.csv")
dbutils.fs.mv(f"{BASE}/landing/orders_1.csv", f"{BASE}/archive/2024-11-01/orders_1.csv")
print("archive:", [f.name for f in dbutils.fs.ls(f"{BASE}/archive/2024-11-01")])
print("landing:", [f.name for f in dbutils.fs.ls(f"{BASE}/landing")])

## 2. Files in volumes are also normal files

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Reads the same files with plain Python and with Spark.<br><br>
<b>Why it matters</b><br>Volumes are mounted as a file system on serverless and recent runtimes, so Python's <code>open</code>, <code>os.listdir</code>, pandas and other libraries work on <code>/Volumes/...</code> paths directly. Use <code>dbutils.fs</code> or Python for small file management, and Spark for reading data at scale.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
The file names from <code>os.listdir</code>, then a Spark DataFrame built from the CSV files left in the landing folder.
</div>

In [0]:
import os
print("os.listdir:", sorted(os.listdir(f"{BASE}/landing")))
spark.read.option("header", True).csv(f"{BASE}/landing").show()

## 3. dbutils.widgets: parameters

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Creates a text widget, a dropdown and a combobox, reads their values, and uses them in a query.<br><br>
<b>Why it matters</b><br>Widgets make notebooks reusable: the same notebook runs for any date or environment. When a job runs the notebook, job or task <b>parameters</b> with the same names fill the widgets automatically. Lesson 04 goes deeper.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
Three widgets at the top of the notebook, and the printed values <code>run_date=2024-11-01, env=dev, country=IN</code>.
</div>

In [0]:
dbutils.widgets.text("run_date", "2024-11-01", "Run date")
dbutils.widgets.dropdown("env", "dev", ["dev", "test", "prod"], "Environment")
dbutils.widgets.combobox("country", "IN", ["IN", "UK", "AE"], "Country")

run_date = dbutils.widgets.get("run_date")
env = dbutils.widgets.get("env")
country = dbutils.widgets.get("country")
print(f"run_date={run_date}, env={env}, country={country}")

## 4. dbutils.secrets: credentials without exposing them

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Lists the secret scopes you can access, and shows how a secret is read and how the notebook redacts it.<br><br>
<b>Why it matters</b><br>Passwords, tokens and keys must never be written in notebooks or Git. Secrets are stored in <b>secret scopes</b>, and <code>dbutils.secrets.get</code> returns the value, which Databricks prints as <code>[REDACTED]</code>. Lesson 05 shows how to create a scope and a secret.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
The list of scopes (possibly empty in a new workspace). If you have run lesson 05, reading its demo secret prints <code>[REDACTED]</code>.
</div>

In [0]:
scopes = [s.name for s in dbutils.secrets.listScopes()]
print("secret scopes:", scopes or "none yet (see lesson 05)")

if "learning" in scopes:
    token = dbutils.secrets.get(scope="learning", key="demo_api_token")
    print("token value:", token)                    # shows [REDACTED]
    print("token length:", len(token))              # you can still use it in code

## 5. dbutils.notebook: run a child notebook and return a value

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Creates a small child notebook next to this one, runs it with parameters, and reads the value it returns with <code>dbutils.notebook.exit</code>.<br><br>
<b>Why it matters</b><br><code>dbutils.notebook.run</code> starts another notebook as a separate run with its own context, passes parameters to its widgets, waits up to a timeout, and returns the string passed to <code>exit</code>. It's the simple way to orchestrate notebooks from a notebook (lesson 07_06). For production, jobs with tasks are preferred.
</div>
<br>
<div style="background:#FFF4DB;border-left:5px solid #E5A100;padding:10px 14px;border-radius:6px;color:#4a3500;">
<b>🆓 Free Edition note</b><br>
The child notebook is created next to this notebook with the Databricks SDK (<code>workspace.import_</code>), from Databricks source format (<code># Databricks notebook source</code>). It's deleted at the end.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
<code>child returned: {"rows": 3, "country": "UK"}</code>.
</div>

In [0]:
import json, base64
from databricks.sdk import WorkspaceClient
from databricks.sdk.service.workspace import ImportFormat, Language

ctx = dbutils.notebook.entry_point.getDbutils().notebook().getContext()
child_path = os.path.dirname(ctx.notebookPath().get()) + "/_child_demo"

child_source = """# Databricks notebook source
dbutils.widgets.text("country", "IN")
country = dbutils.widgets.get("country")
rows = spark.range(3).count()

# COMMAND ----------

import json
dbutils.notebook.exit(json.dumps({"rows": rows, "country": country}))
"""
w = WorkspaceClient()
w.workspace.import_(path=child_path, format=ImportFormat.SOURCE, language=Language.PYTHON,
                    content=base64.b64encode(child_source.encode()).decode(), overwrite=True)

result = dbutils.notebook.run(child_path, 120, {"country": "UK"})
print("child returned:", result)
print("parsed:", json.loads(result))
w.workspace.delete(child_path)

## 6. dbutils.jobs.taskValues: pass values between job tasks

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Sets task values that later tasks of the same job can read, and shows how a downstream task reads them.<br><br>
<b>Why it matters</b><br>Tasks in a job run in separate contexts, so Python variables don't carry over. Task values pass small results (row counts, file names, flags) to downstream tasks, which can also use them in conditions (<code>if/else</code> tasks) and parameters such as <code>{{tasks.load.values.rows_loaded}}</code>. Outside a job run, <code>set</code> is ignored and <code>get</code> returns the <code>debugValue</code>.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
No error when setting values. Reading returns the <code>debugValue</code> (42) because this notebook isn't running inside a job.
</div>

In [0]:
dbutils.jobs.taskValues.set(key="rows_loaded", value=42)
dbutils.jobs.taskValues.set(key="files", value=["orders_0.csv", "orders_2.csv"])

# In a downstream task named e.g. "report", you would read the value set by the task "load":
rows = dbutils.jobs.taskValues.get(taskKey="load", key="rows_loaded", default=0, debugValue=42)
print("rows_loaded (debug value outside a job):", rows)

## 7. Cleanup

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Removes the widgets and the lesson's files.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
No widgets left, and <code>True</code> for the deleted folder.
</div>

In [0]:
dbutils.widgets.removeAll()
print(dbutils.fs.rm(BASE, recurse=True))

## Under the hood

```
dbutils (in the notebook process)
  fs        -> file system client: volumes (/Volumes), cloud URIs (abfss://, s3://), workspace files
  widgets   -> notebook UI parameters; filled from job/task parameters at run time
  secrets   -> calls the secrets service; values are redacted in notebook output
  notebook  -> run(): submits a separate ephemeral run and waits; exit(): ends this run with a value
  jobs.taskValues -> stored with the job run, readable by downstream tasks
```

<span style="background:#1E9E5A;color:white;padding:2px 10px;border-radius:12px;font-size:12px;">Metadata only</span> `dbutils` calls run on the driver (or the Python client on serverless), not on executors. Don't call `dbutils` inside UDFs.

`dbutils` exists only on Databricks. For code that must also run outside Databricks (unit tests, local development), wrap it in small functions you can replace, or use the **Databricks SDK** (`WorkspaceClient().dbutils`), which works from anywhere with credentials.

In [0]:
# A pattern for code that runs both on Databricks and locally (tests):
def get_param(name, default):
    try:
        return dbutils.widgets.get(name)          # noqa: F821  (only defined on Databricks)
    except Exception:
        return default

print("run_date:", get_param("run_date", "2024-11-01"))

## Common mistakes and troubleshooting log

<div style="background:#FDEAEA;border-left:5px solid #D64545;padding:10px 14px;border-radius:6px;color:#4a1414;">
<b>⛔ Problem: <code>NameError: name 'dbutils' is not defined</code></b><br>
You're outside a Databricks notebook (local Python, a library module). Pass values as parameters, or use <code>WorkspaceClient().dbutils</code>.<br><br>
**⛔ Problem: <code>dbutils.fs.ls("dbfs:/...")</code> fails on Free Edition**<br>The DBFS root isn't available. Use <code>/Volumes/&lt;catalog&gt;/&lt;schema&gt;/&lt;volume&gt;/...</code>.<br><br>
**⛔ Problem: the secret prints as <code>[REDACTED]</code>**<br>That's intended. Use the value in code. Don't try to work around the redaction.<br><br>
**⛔ Problem: <code>dbutils.notebook.run</code> times out**<br>The timeout (seconds) is too short, or the child is stuck. A timeout of 0 means no timeout.<br><br>
<b>⛔ Problem: task values not found downstream</b><br>The upstream task key or value key is misspelled, or the upstream task didn't run.<br><br>
<b>✍️ My own problems</b><br>(add yours here)
</div>

## Interview Q&A

<div style="background:#F1E9FF;border-left:5px solid #7A4DD8;padding:10px 14px;border-radius:6px;color:#2d1a57;">
<b>🎤 1. What is <code>dbutils</code>?</b><br>
The Databricks utilities available in notebooks, for file operations (<code>fs</code>), parameters (<code>widgets</code>), secrets (<code>secrets</code>), notebook orchestration (<code>notebook</code>) and passing values between job tasks (<code>jobs.taskValues</code>).<br><br>

<b>🎤 2. How do you pass parameters into a notebook?</b><br>
Define widgets with <code>dbutils.widgets.text</code> or <code>dropdown</code> and read them with <code>dbutils.widgets.get</code>. When a job runs the notebook, job or task parameters with the same names fill the widgets.<br><br>

<b>🎤 3. How do you use credentials in a notebook securely?</b><br>
Store them in a secret scope and read them with <code>dbutils.secrets.get(scope, key)</code>. The value is redacted in outputs and never appears in code or Git.<br><br>

<b>🎤 4. What is the difference between <code>%run</code> and <code>dbutils.notebook.run</code>?</b><br>
<code>%run</code> executes another notebook inline in the same context, with no parameters or return value. <code>dbutils.notebook.run</code> starts it as a separate run with parameters and a timeout, and returns the value passed to <code>dbutils.notebook.exit</code>.<br><br>

<b>🎤 5. How do tasks in a job share information?</b><br>
Through task values: <code>dbutils.jobs.taskValues.set</code> in one task and <code>get</code> in a downstream task, or references such as <code>{{tasks.&lt;task&gt;.values.&lt;key&gt;}}</code> in parameters and conditions. For larger data, through tables.
</div>

## Certification corner

<div style="background:#E6F6F7;border-left:5px solid #0E8C8C;padding:10px 14px;border-radius:6px;color:#0b3536;">
<b>🎓 Exam-style questions (Data Engineer Associate)</b><br><br>
<b>Q1. A notebook task needs to make the number of rows it loaded available to a later task in the same job. What should it use?</b><br>
A. A global Python variable<br>
B. <code>dbutils.jobs.taskValues.set(key="rows", value=n)</code><br>
C. <code>dbutils.widgets.text("rows", n)</code><br>
D. <code>spark.conf.set("rows", n)</code><br>
<details><summary><b>Show answer</b></summary><b>B.</b></details><br><br>

<b>Q2. Which command lists the files in a Unity Catalog volume directory?</b><br>
A. <code>dbutils.fs.ls("/Volumes/main/raw/files/landing")</code><br>
B. <code>spark.catalog.listFiles("/Volumes/...")</code><br>
C. <code>dbutils.volumes.list(...)</code><br>
D. <code>SHOW FILES IN main.raw.files</code><br>
<details><summary><b>Show answer</b></summary><b>A.</b> <code>LIST '/Volumes/...'</code> is the SQL equivalent.</details><br><br>

<b>Q3. What does <code>dbutils.notebook.exit("done")</code> do?</b><br>
A. Stops the cluster<br>
B. Ends the notebook run and returns <code>"done"</code> to the caller<br>
C. Deletes the notebook's widgets<br>
D. Commits the notebook to Git<br>
<details><summary><b>Show answer</b></summary><b>B.</b></details>
</div>

## Practice problem

<div style="background:#FFEFE5;border-left:5px solid #F07D2E;padding:10px 14px;border-radius:6px;color:#4a2300;">
<b>🧪 Without copying from above:</b>
<ol><li>Create a folder <code>practice/</code> in the lesson volume path with 5 small text files, list them sorted by name, and print the total size</li><li>Move the files whose names end in an even number to <code>practice/even/</code></li><li>Create a dropdown widget <code>mode</code> with values <code>full</code> and <code>incremental</code> and print a different message for each</li><li>Write a helper <code>param(name, default)</code> that reads a widget and falls back to a default when the widget doesn't exist</li></ol>
</div>

In [0]:
# Your solution for the practice problem goes here.
# Add a comment above each step explaining what it does.

## Solution

<div style="background:#FFF4DB;border-left:5px solid #E5A100;padding:10px 14px;border-radius:6px;color:#4a3500;">
<b>⚠️ Try it yourself first</b><br>
Only look at this after you have a working answer of your own.
</div>

In [0]:
BASE = "/Volumes/workspace/platform/files/lesson03"

# 1. Create, list, total size
for i in range(5):
    dbutils.fs.put(f"{BASE}/practice/file_{i}.txt", "x" * (i + 1) * 10, overwrite=True)
files = sorted(dbutils.fs.ls(f"{BASE}/practice"), key=lambda f: f.name)
print([f.name for f in files], "| total bytes:", sum(f.size for f in files))

# 2. Move the even-numbered files
for f in files:
    if f.name.endswith((".txt",)) and int(f.name.split("_")[1].split(".")[0]) % 2 == 0:
        dbutils.fs.mv(f.path, f"{BASE}/practice/even/{f.name}")
print("even:", [f.name for f in dbutils.fs.ls(f"{BASE}/practice/even")])

# 3. Dropdown-driven behaviour
dbutils.widgets.dropdown("mode", "incremental", ["full", "incremental"])
print("Rebuilding everything" if dbutils.widgets.get("mode") == "full" else "Loading only new data")

# 4. Widget with fallback
def param(name, default):
    try:
        return dbutils.widgets.get(name)
    except Exception:
        return default
print(param("mode", "incremental"), param("does_not_exist", "fallback"))

dbutils.widgets.removeAll()
dbutils.fs.rm(BASE, recurse=True)

## Summary and cheat sheet

<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Key takeaways</b>
<ul><li><code>dbutils.fs</code>: list, read, copy, move and delete files in volumes and cloud paths</li><li><code>dbutils.widgets</code>: notebook parameters, filled by job parameters at run time</li><li><code>dbutils.secrets</code>: credentials from secret scopes, redacted in output</li><li><code>dbutils.notebook.run</code> / <code>exit</code>: run a child notebook with parameters and get a value back</li><li><code>dbutils.jobs.taskValues</code>: pass small values between job tasks</li><li><code>dbutils</code> runs on the driver/client only. Use volumes, not DBFS root, on Free Edition</li></ul>
</div>

| Task | Code |
|---|---|
| List | `dbutils.fs.ls(path)` |
| Write small file | `dbutils.fs.put(path, text, overwrite=True)` |
| Copy / move / delete | `dbutils.fs.cp(a, b)`, `.mv(a, b)`, `.rm(p, recurse=True)` |
| Widget | `dbutils.widgets.text("k", "default")`, `.get("k")` |
| Secret | `dbutils.secrets.get(scope="s", key="k")` |
| Run notebook | `dbutils.notebook.run("./child", 600, {"k": "v"})` |
| Task value | `dbutils.jobs.taskValues.set(key="k", value=v)` |

## Git commit

```
git add 04_databricks/03_dbutils.ipynb
git commit -m "add 04_03 dbutils notebook"
```